In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from faker import Faker
import pandas as pd
import random

In [2]:
# 1. Connexion à Spark
spark = SparkSession.builder \
    .appName("Generateur_Data_Reelle") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

fake = Faker('en_US')

In [3]:
# ==========================================
# ÉTAPE 1 : Génération des Dimensions
# ==========================================
print("Génération des Aéroports, Transporteurs et Avions avec des données réelles...")

airports_data = []
for i in range(1, 3401):
    airports_data.append({
        "iata_code": fake.lexify(text='???').upper(),
        "nom": fake.company() + " Airport",
        "ville": fake.city(),
        "etat": fake.state_abbr(),
        "latitude": round(float(fake.latitude()), 4),
        "longitude": round(float(fake.longitude()), 4),
        "airport_id": i
    })
airports = spark.createDataFrame(pd.DataFrame(airports_data))

carriers_data = []
for i in range(1, 1501):
    carriers_data.append({
        "carrier_code": fake.lexify(text='??').upper(),
        "nom": fake.company() + " Airlines",
        "carrier_id": i
    })
carriers = spark.createDataFrame(pd.DataFrame(carriers_data))

constructeurs = ["Boeing", "Airbus", "Embraer", "Bombardier", "Cessna"]
planes_data = []
for i in range(1, 5001):
    planes_data.append({
        "immatriculation": "N" + str(fake.random_int(min=100, max=9999)) + fake.lexify(text='??').upper(),
        "constructeur": random.choice(constructeurs),
        "modele": fake.bothify(text='7##-200' if random.random() > 0.5 else 'A3##'),
        "annee": fake.random_int(min=1987, max=2008),
        "plane_id": i
    })
planes = spark.createDataFrame(pd.DataFrame(planes_data))


Génération des Aéroports, Transporteurs et Avions avec des données réelles...


In [4]:
# ==========================================
# ÉTAPE 2 : Génération de la Table Flights
# ==========================================
print("Génération des vols...")
NUM_LIGNES = 10000000

flights = spark.range(NUM_LIGNES).select(
    (F.floor(F.rand() * 22) + 1987).alias("Year"),
    (F.floor(F.rand() * 12) + 1).alias("Month"),
    (F.floor(F.rand() * 28) + 1).alias("DayofMonth"),
    (F.floor(F.rand() * 2400)).alias("DepTime"),
    (F.floor(F.rand() * 1500) + 1).alias("carrier_id"),
    (F.floor(F.rand() * 5000) + 1).alias("plane_id"),
    (F.floor(F.rand() * 3400) + 1).alias("origin_id"),
    (F.floor(F.rand() * 3400) + 1).alias("dest_id"),
    F.round(F.rand() * 100 - 20, 0).alias("ArrDelay"),
    F.round(F.rand() * 100 - 20, 0).alias("DepDelay"),
    F.round(F.rand() * 2000 + 100, 0).alias("Distance"),
    F.floor(F.rand() * 1.05).alias("Cancelled")
)



Génération des vols...


In [5]:
# ==========================================
# ÉTAPE 3 : Sauvegarde dans HDFS
# ==========================================
print("Sauvegarde des données dans HDFS...")
airports.write.mode("overwrite").csv("hdfs://namenode:9000/data/airports", header=True)
carriers.write.mode("overwrite").csv("hdfs://namenode:9000/data/carriers", header=True)
planes.write.mode("overwrite").csv("hdfs://namenode:9000/data/planes", header=True)
flights.write.mode("overwrite").csv("hdfs://namenode:9000/data/flights", header=True)

print("Les données ont été sauvegardées avec succès avec des noms réels !")

Sauvegarde des données dans HDFS...
Les données ont été sauvegardées avec succès avec des noms réels !


In [8]:
# Côté client
!python -c "import pyspark; print(pyspark.__version__)"



3.5.0


In [10]:
# Côté cluster (dans le conteneur spark-master)
!spark-submit --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.5.0
      /_/
                        
Using Scala version 2.12.18, OpenJDK 64-Bit Server VM, 11.0.32
Branch HEAD
Compiled by user ubuntu on 2023-09-09T01:53:20Z
Revision ce5ddad990373636e94071e7cef2f31021add07b
Url https://github.com/apache/spark
Type --help for more information.


In [12]:
# Côté cluster : vérifier le chemin des jars Spark
!find / -name "spark-core*.jar" 2>/dev/null

/usr/local/spark-3.5.0-bin-hadoop3/jars/spark-core_2.12-3.5.0.jar
/opt/conda/lib/python3.11/site-packages/pyspark/jars/spark-core_2.12-3.5.1.jar


In [13]:
# Côté client : vérifier les jars pyspark
!python -c "import pyspark, os; print(os.path.dirname(pyspark.__file__))"

/usr/local/spark/python/pyspark
